# 01 - Source Discovery and Public Data Ingestion

This notebook performs initial discovery and ingestion of public Israeli urban-renewal-related data sources for the **Urban Renewal Legal Scout - Birthday MVP**.

The notebook creates the project folder structure, searches official/public data portals, tests public accessibility, downloads relevant datasets when possible, stores raw files, writes metadata, inspects loaded datasets, and creates a raw collected CSV for Notebook 02.

**Disclaimer:** This notebook collects public information only. It does not provide legal advice, planning advice, real estate advice, or binding predictions.

## 1 - Imports and Global Configuration

Import the core libraries, configure paths, create the expected folder structure, and define runtime flags.

In [41]:
from __future__ import annotations

import io
import json
import re
import traceback
import warnings
import zipfile
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

import numpy as np
import pandas as pd
import requests

try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except Exception:
    gpd = None
    HAS_GEOPANDAS = False

warnings.filterwarnings("default")

# Make the notebook runnable from either the project root or the notebooks folder.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = DATA_DIR / "metadata"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MANUAL_SOURCES_DIR = DATA_DIR / "manual_sources"

for folder in [DATA_DIR, RAW_DIR, PROCESSED_DIR, METADATA_DIR, MANUAL_SOURCES_DIR, OUTPUTS_DIR, PROJECT_ROOT / "notebooks", PROJECT_ROOT / "app", PROJECT_ROOT / "models"]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_DATA_GOV_SEARCH = True
RUN_DIRECT_DOWNLOADS = True
RUN_OPTIONAL_GIS_CHECKS = True
REQUEST_TIMEOUT = 30

LAST_CHECKED = datetime.now().isoformat(timespec="seconds")

DEFAULT_REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36 UrbanRenewalLegalScoutMVP/0.1",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "he-IL,he;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://data.gov.il/",
}
REQUEST_SESSION = requests.Session()
REQUEST_SESSION.headers.update(DEFAULT_REQUEST_HEADERS)

print(f"Project root: {PROJECT_ROOT}")
print(f"Last checked: {LAST_CHECKED}")
print(f"geopandas available: {HAS_GEOPANDAS}")

Project root: C:\Users\Guy\Desktop\Birthday present
Last checked: 2026-05-26T15:05:10
geopandas available: False


## 2 - Helper Functions

These helpers keep network access, file naming, loading, inspection, and inventory records consistent. Each source is allowed to fail independently so the notebook can continue.

In [42]:
def safe_request(
    url: str,
    method: str = "GET",
    params: Optional[dict] = None,
    headers: Optional[dict] = None,
    timeout: int = 30,
    json_payload: Optional[dict] = None,
) -> Dict[str, Any]:
    """Run a requests call and return a structured result without raising."""
    result = {
        "success": False,
        "status_code": None,
        "content_type": None,
        "text_preview": None,
        "response_object": None,
        "error_message": None,
    }
    try:
        request_headers = dict(globals().get("DEFAULT_REQUEST_HEADERS", {}))
        if not request_headers:
            request_headers = {
                "User-Agent": "Mozilla/5.0 UrbanRenewalLegalScoutMVP/0.1",
                "Accept": "application/json, text/plain, */*",
                "Accept-Language": "he-IL,he;q=0.9,en-US;q=0.8,en;q=0.7",
                "Referer": "https://data.gov.il/",
            }
        if headers:
            request_headers.update(headers)
        session = globals().get("REQUEST_SESSION") or requests.Session()
        response = session.request(
            method=method.upper(),
            url=url,
            params=params,
            json=json_payload,
            headers=request_headers,
            timeout=timeout,
        )
        content_type = response.headers.get("content-type", "")
        result.update(
            {
                "success": 200 <= response.status_code < 400,
                "status_code": response.status_code,
                "content_type": content_type,
                "text_preview": response.text[:500] if "text" in content_type.lower() or "json" in content_type.lower() else None,
                "response_object": response,
                "error_message": None if 200 <= response.status_code < 400 else f"HTTP {response.status_code}",
            }
        )
    except Exception as exc:
        result["error_message"] = f"{type(exc).__name__}: {exc}"
    return result

def sanitize_filename(name: Any) -> str:
    """Convert an arbitrary source/resource name into a filesystem-safe stem."""
    text = str(name or "source").strip()
    text = re.sub(r"[\\/:*?\"<>|]+", "_", text)
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text).strip("._ ")
    return text[:120] or "source"


def infer_file_extension(url: str, content_type: Optional[str] = None, fallback: str = ".csv") -> str:
    """Infer a likely file extension from a URL or HTTP content type."""
    clean_url = str(url or "").split("?", 1)[0].lower()
    for ext in [".geojson", ".json", ".xlsx", ".xls", ".csv", ".zip"]:
        if clean_url.endswith(ext):
            return ext

    ct = (content_type or "").lower()
    if "geo+json" in ct or "geojson" in ct:
        return ".geojson"
    if "json" in ct:
        return ".json"
    if "spreadsheetml" in ct or "excel" in ct:
        return ".xlsx"
    if "csv" in ct or "comma-separated" in ct:
        return ".csv"
    if "zip" in ct or "compressed" in ct:
        return ".zip"
    return fallback


def _decode_preview(data: bytes, limit: int = 2_000) -> str:
    """Decode a small byte preview with common Hebrew-friendly encodings."""
    sample = (data or b"")[:limit]
    for encoding in ["utf-8-sig", "utf-8", "cp1255", "windows-1255", "iso-8859-8"]:
        try:
            return sample.decode(encoding, errors="ignore")
        except Exception:
            continue
    return sample.decode("utf-8", errors="ignore")


API_DIAGNOSTICS_COLUMNS = [
    "endpoint",
    "method",
    "search_term",
    "status_code",
    "content_type",
    "response_preview",
    "parsed_json_success",
    "error_message",
]


def add_api_diagnostic(
    records: List[Dict[str, Any]],
    endpoint: str,
    method: str,
    search_term: Optional[str] = None,
    response_result: Optional[Dict[str, Any]] = None,
    parsed_json_success: Optional[bool] = None,
    error_message: Optional[str] = None,
) -> None:
    """Append a compact diagnostic row for API troubleshooting."""
    response_result = response_result or {}
    response = response_result.get("response_object")
    preview = response_result.get("text_preview")
    if preview is None and response is not None:
        try:
            preview = _decode_preview(response.content)[:500]
        except Exception:
            preview = None
    records.append(
        {
            "endpoint": endpoint,
            "method": method.upper(),
            "search_term": search_term,
            "status_code": response_result.get("status_code"),
            "content_type": response_result.get("content_type"),
            "response_preview": preview,
            "parsed_json_success": parsed_json_success,
            "error_message": error_message or response_result.get("error_message"),
        }
    )


def save_api_diagnostics() -> pd.DataFrame:
    diagnostics_df = pd.DataFrame(api_diagnostics_records, columns=API_DIAGNOSTICS_COLUMNS)
    diagnostics_path = METADATA_DIR / "api_diagnostics.csv"
    diagnostics_df.to_csv(diagnostics_path, index=False, encoding="utf-8-sig")
    print(f"Saved API diagnostics: {diagnostics_path.relative_to(PROJECT_ROOT)} ({len(diagnostics_df):,} rows)")
    return diagnostics_df


HTML_OR_BLOCKED_MARKERS = [
    "<html",
    "<!doctype html",
    "<script",
    "security violation",
    "captcha",
    "access denied",
    "forbidden",
    "verify you are human",
    "cloudflare",
    "akamai",
    "request unsuccessful",
    "blocked",
]


def content_looks_like_html_or_security_page(preview: Any) -> bool:
    text = str(preview or "").strip().lower()
    return any(marker in text for marker in HTML_OR_BLOCKED_MARKERS)


def _matched_keywords(text: Any, keywords: Iterable[str]) -> List[str]:
    text_norm = str(text or "").lower()
    return [keyword for keyword in keywords if str(keyword).lower() in text_norm]


def detect_content_kind(response: requests.Response, expected_format: Optional[str] = None, source_url: Optional[str] = None) -> str:
    """Classify downloaded bytes before pandas sees them."""
    data = response.content or b""
    content_type = response.headers.get("content-type", "").lower()
    url = str(source_url or response.url or "").split("?", 1)[0].lower()
    expected = str(expected_format or "").strip().upper()
    preview = _decode_preview(data)
    preview_l = preview.strip().lower()

    if not data:
        return "unknown"
    if content_looks_like_html_or_security_page(preview_l):
        blocked_terms = ["security violation", "captcha", "access denied", "forbidden", "verify you are human", "cloudflare", "akamai", "blocked"]
        return "blocked_or_security_page" if any(term in preview_l for term in blocked_terms) else "html"
    if data[:4] == b"PK\x03\x04":
        if expected in {"XLSX", "EXCEL"} or url.endswith(".xlsx") or "spreadsheetml" in content_type:
            return "excel"
        return "zip"
    if data[:8] == b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1":
        return "excel"
    if preview_l.startswith("{") or preview_l.startswith("[") or "json" in content_type:
        try:
            obj = json.loads(preview if len(data) <= 2_000 else data.decode("utf-8-sig", errors="ignore"))
            if isinstance(obj, dict) and obj.get("type") == "FeatureCollection":
                return "geojson"
            return "json"
        except Exception:
            if "json" in content_type:
                return "json"
    if "geojson" in content_type or url.endswith(".geojson"):
        return "geojson"
    if "csv" in content_type or url.endswith(".csv"):
        return "csv"
    if "excel" in content_type or "spreadsheet" in content_type or url.endswith((".xls", ".xlsx")):
        return "excel"
    if "zip" in content_type or url.endswith(".zip"):
        return "zip"
    if "text/html" in content_type:
        return "html"
    if ("," in preview or ";" in preview or "\t" in preview) and "\n" in preview:
        return "csv"
    return "unknown"


def validate_response_content(
    response: requests.Response,
    expected_format: Optional[str] = None,
    source_url: Optional[str] = None,
) -> Dict[str, Any]:
    """Validate that a successful HTTP response appears to contain a real dataset."""
    data = response.content or b""
    content_preview = _decode_preview(data)
    detected_content_kind = detect_content_kind(response, expected_format=expected_format, source_url=source_url)
    expected = str(expected_format or "").strip().upper()

    result = {
        "is_valid_download": False,
        "detected_content_kind": detected_content_kind,
        "validation_error": None,
        "content_preview": content_preview[:500],
    }

    if not data:
        result["validation_error"] = "Empty response body."
        return result
    if detected_content_kind in {"html", "blocked_or_security_page"} or content_looks_like_html_or_security_page(content_preview):
        result["validation_error"] = "Downloaded content appears to be HTML/security page, not a dataset."
        return result

    compatible_kinds = {
        "CSV": {"csv"},
        "JSON": {"json", "geojson"},
        "GEOJSON": {"geojson", "json"},
        "XLSX": {"excel"},
        "XLS": {"excel"},
        "EXCEL": {"excel"},
        "ZIP": {"zip", "excel"},
    }
    if expected in compatible_kinds and detected_content_kind not in compatible_kinds[expected]:
        result["validation_error"] = f"Response content kind '{detected_content_kind}' is not compatible with expected format '{expected}'."
        return result
    if expected not in compatible_kinds and detected_content_kind == "unknown":
        result["validation_error"] = "Could not identify response as a supported dataset format."
        return result

    result["is_valid_download"] = True
    return result


def save_response_content(response: requests.Response, target_path: Path) -> Path:
    """Save response bytes to disk."""
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_bytes(response.content)
    return target_path


def _read_csv_bytes(data: bytes) -> pd.DataFrame:
    preview = _decode_preview(data)
    if content_looks_like_html_or_security_page(preview):
        raise ValueError("Downloaded content appears to be HTML/security page, not a dataset.")
    encodings = ["utf-8-sig", "utf-8", "cp1255", "windows-1255", "iso-8859-8"]
    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(io.BytesIO(data), encoding=encoding, sep=None, engine="python", on_bad_lines="skip")
        except Exception as exc:
            last_error = exc
    raise last_error if last_error else ValueError("CSV loading failed")


def _json_to_dataframe(obj: Any) -> pd.DataFrame:
    if isinstance(obj, list):
        return pd.json_normalize(obj)
    if isinstance(obj, dict):
        if obj.get("type") == "FeatureCollection" and isinstance(obj.get("features"), list):
            return pd.json_normalize(obj["features"])
        for key, value in obj.items():
            if isinstance(value, list) and value and isinstance(value[0], (dict, list)):
                return pd.json_normalize(value)
        return pd.json_normalize(obj)
    raise ValueError("Unsupported JSON structure")


def try_load_dataset(file_path: Path) -> Dict[str, Any]:
    """Attempt to load a saved file into pandas. ZIP files are inspected for loadable tabular resources."""
    file_path = Path(file_path)
    result = {"success": False, "df": None, "error_message": None}
    try:
        suffix = file_path.suffix.lower()
        first_bytes = file_path.read_bytes()[:2_000]
        if content_looks_like_html_or_security_page(_decode_preview(first_bytes)):
            raise ValueError("Downloaded content appears to be HTML/security page, not a dataset.")

        if suffix == ".csv":
            df = _read_csv_bytes(file_path.read_bytes())
        elif suffix in [".json", ".geojson"]:
            try:
                obj = json.loads(file_path.read_text(encoding="utf-8-sig"))
            except UnicodeDecodeError:
                obj = json.loads(file_path.read_text(encoding="cp1255"))
            df = _json_to_dataframe(obj)
        elif suffix in [".xlsx", ".xls"]:
            df = pd.read_excel(file_path)
        elif suffix == ".zip":
            df = None
            with zipfile.ZipFile(file_path) as zf:
                names = [name for name in zf.namelist() if not name.endswith("/")]
                supported = [name for name in names if Path(name).suffix.lower() in [".csv", ".xlsx", ".xls", ".json", ".geojson"]]
                if supported:
                    member = supported[0]
                    data = zf.read(member)
                    if content_looks_like_html_or_security_page(_decode_preview(data)):
                        raise ValueError("Downloaded content appears to be HTML/security page, not a dataset.")
                    member_suffix = Path(member).suffix.lower()
                    if member_suffix == ".csv":
                        df = _read_csv_bytes(data)
                    elif member_suffix in [".json", ".geojson"]:
                        obj = json.loads(data.decode("utf-8-sig"))
                        df = _json_to_dataframe(obj)
                    else:
                        df = pd.read_excel(io.BytesIO(data))
                elif HAS_GEOPANDAS and any(Path(name).suffix.lower() == ".shp" for name in names):
                    gdf = gpd.read_file(f"zip://{file_path}")
                    df = pd.DataFrame(gdf)
                else:
                    raise ValueError(f"ZIP contains no supported tabular files. Members: {names[:20]}")
        else:
            raise ValueError(f"Unsupported file extension: {suffix}")

        result.update({"success": True, "df": df, "error_message": None})
    except Exception as exc:
        result["error_message"] = f"{type(exc).__name__}: {exc}"
    return result


def inspect_dataframe(df: pd.DataFrame) -> Dict[str, Any]:
    """Return compact dataframe metadata and a small sample."""
    if df is None:
        return {"num_rows": 0, "num_columns": 0, "columns": [], "missing_ratio_mean": None, "sample_records": []}
    safe_sample = df.head(3).replace({np.nan: None}).astype(object).to_dict(orient="records")
    safe_sample = json.loads(json.dumps(safe_sample, ensure_ascii=False, default=str))
    return {
        "num_rows": int(len(df)),
        "num_columns": int(df.shape[1]),
        "columns": [str(col) for col in df.columns],
        "missing_ratio_mean": float(df.isna().mean().mean()) if df.shape[1] else None,
        "sample_records": safe_sample,
    }


VALID_ACCESS_LEVELS = {
    "PUBLIC_DOWNLOAD",
    "PUBLIC_API",
    "PUBLIC_BUT_FRAGILE",
    "PUBLIC_MANUAL_ONLY",
    "LOGIN_REQUIRED",
    "NOT_RELEVANT",
    "FAILED",
    "UNKNOWN",
}


def classify_access_level(
    success: Optional[bool] = None,
    source_type: Optional[str] = None,
    access_method: Optional[str] = None,
    download_status: Optional[str] = None,
    notes: Optional[str] = None,
) -> str:
    """Classify source access using a constrained vocabulary."""
    text = " ".join([str(source_type or ""), str(access_method or ""), str(download_status or ""), str(notes or "")]).lower()
    if "login" in text or "הרשמה" in text:
        return "LOGIN_REQUIRED"
    if success is False or download_status == "failed":
        return "FAILED"
    if "api" in text:
        return "PUBLIC_API" if success else "UNKNOWN"
    if "download" in text or download_status == "downloaded":
        return "PUBLIC_DOWNLOAD" if success else "UNKNOWN"
    if "fragile" in text or "dashboard" in text or "gis" in text or "map" in text:
        return "PUBLIC_BUT_FRAGILE" if success else "UNKNOWN"
    if "manual" in text:
        return "PUBLIC_MANUAL_ONLY"
    return "UNKNOWN"


def _json_or_empty(value: Optional[Iterable[str] | str]) -> str:
    if value is None:
        return json.dumps([], ensure_ascii=False)
    if isinstance(value, str):
        return value
    return json.dumps(list(value), ensure_ascii=False)


def add_inventory_record(
    records: List[Dict[str, Any]],
    source_name: str,
    source_url: str,
    source_type: str,
    priority: int,
    expected_format: str,
    access_method: str,
    access_level: str = "UNKNOWN",
    download_status: str = "not_attempted",
    http_status_code: Optional[int] = None,
    content_type: Optional[str] = None,
    file_path: Optional[Path | str] = None,
    load_status: str = "not_attempted",
    num_rows: Optional[int] = None,
    num_columns: Optional[int] = None,
    columns: Optional[Iterable[str]] = None,
    error_message: Optional[str] = None,
    notes: Optional[str] = None,
    load_method: Optional[str] = None,
    content_validation_status: Optional[str] = None,
    content_validation_error: Optional[str] = None,
    detected_content_kind: Optional[str] = None,
    is_valid_dataset: Optional[bool] = None,
    is_relevant_dataset: Optional[bool] = None,
    strict_urban_renewal_match: Optional[bool] = None,
    matched_strict_keywords: Optional[Iterable[str] | str] = None,
    final_included_in_raw_csv: bool = False,
    original_resource_id: Optional[str] = None,
    total_records_reported: Optional[int] = None,
    total_records_loaded: Optional[int] = None,
) -> None:
    access_level = access_level if access_level in VALID_ACCESS_LEVELS else "UNKNOWN"
    records.append(
        {
            "source_name": source_name,
            "source_url": source_url,
            "source_type": source_type,
            "priority": priority,
            "expected_format": expected_format,
            "access_method": access_method,
            "access_level": access_level,
            "download_status": download_status,
            "http_status_code": http_status_code,
            "content_type": content_type,
            "file_path": str(Path(file_path).relative_to(PROJECT_ROOT)) if file_path else None,
            "load_status": load_status,
            "num_rows": num_rows,
            "num_columns": num_columns,
            "columns": json.dumps(list(columns or []), ensure_ascii=False),
            "last_checked": LAST_CHECKED,
            "error_message": error_message,
            "notes": notes,
            "load_method": load_method,
            "content_validation_status": content_validation_status,
            "content_validation_error": content_validation_error,
            "detected_content_kind": detected_content_kind,
            "is_valid_dataset": is_valid_dataset,
            "is_relevant_dataset": is_relevant_dataset,
            "strict_urban_renewal_match": strict_urban_renewal_match,
            "matched_strict_keywords": _json_or_empty(matched_strict_keywords),
            "final_included_in_raw_csv": bool(final_included_in_raw_csv),
            "original_resource_id": original_resource_id,
            "total_records_reported": total_records_reported,
            "total_records_loaded": total_records_loaded,
        }
    )


def text_contains_any(text: Any, keywords: Iterable[str]) -> bool:
    text_norm = str(text or "").lower()
    return any(str(keyword).lower() in text_norm for keyword in keywords)


def detect_relevance_reason(row_or_text: Any, keywords: Iterable[str]) -> str:
    text = str(row_or_text or "").lower()
    hits = [keyword for keyword in keywords if str(keyword).lower() in text]
    return "; ".join([f"matched keyword: {hit}" for hit in hits[:8]]) if hits else "metadata/columns selected as relevant candidate"


def has_keyword_in_columns(columns: Iterable[str], keywords: Iterable[str]) -> bool:
    joined = " | ".join(str(col).lower() for col in columns)
    return any(str(keyword).lower() in joined for keyword in keywords)


def dataframe_looks_like_html(df: pd.DataFrame) -> bool:
    """Catch HTML/security pages that pandas parsed into a rectangular table."""
    if df is None or df.empty:
        return False
    column_names = [str(col) for col in df.columns]
    first_col = column_names[0] if column_names else ""
    joined_columns = " | ".join(column_names[:20])
    first_values = " | ".join(str(value) for value in df.head(1).astype(str).values.flatten()[:20])
    combined_preview = f"{first_col} | {joined_columns} | {first_values}"

    if str(first_col).strip().lower().startswith(("<html", "<!doctype html")):
        return True
    if any(len(col) > 200 and content_looks_like_html_or_security_page(col) for col in column_names):
        return True
    if df.shape[0] == 1 and df.shape[1] > 50 and content_looks_like_html_or_security_page(combined_preview):
        return True
    return content_looks_like_html_or_security_page(combined_preview)


def dataset_has_relevant_columns(columns: Iterable[str]) -> bool:
    return has_keyword_in_columns(columns, DATASET_COLUMN_RELEVANCE_KEYWORDS)


## 3 - Define Initial Source Registry

The registry includes directly searchable APIs and public sources that should be checked without deep scraping or login-based workflows.

In [43]:
DATA_GOV_CKAN_ENDPOINT = "https://data.gov.il/api/3/action/package_search"
DATA_GOV_DATASTORE_ENDPOINT = "https://data.gov.il/api/3/action/datastore_search"

SEARCH_TERMS = [
    "התחדשות עירונית",
    "פינוי בינוי",
    "תמ\"א 38",
    "מתחמי התחדשות",
    "רשות ממשלתית להתחדשות עירונית",
    "urban renewal",
    "renewal",
    "housing",
    "תכנון",
    "תכנית",
    "תוכניות",
]

STRICT_URBAN_RENEWAL_KEYWORDS = [
    "התחדשות עירונית",
    "התחדשות",
    "פינוי בינוי",
    "פינוי-בינוי",
    "בינוי פינוי",
    "תמ\"א 38",
    "תמ״א 38",
    "תמא 38",
    "מתחמי התחדשות",
    "מתחם התחדשות",
    "מתחמים מוכרזים",
    "רשות ממשלתית להתחדשות עירונית",
    "urban renewal",
    "redevelopment",
]

BROAD_RELATED_KEYWORDS = [
    "תכנית",
    "תוכנית",
    "תכנון",
    "דיור",
    "housing",
    "planning",
    "plan",
    "units",
    "construction",
]

EXPLICIT_URBAN_RENEWAL_SOURCE_KEYWORDS = [
    "רשות ממשלתית להתחדשות עירונית",
    "הרשות הממשלתית להתחדשות עירונית",
    "הרשות להתחדשות עירונית",
    "urban renewal authority",
]

DATASET_COLUMN_RELEVANCE_KEYWORDS = STRICT_URBAN_RENEWAL_KEYWORDS + BROAD_RELATED_KEYWORDS + [
    "city",
    "עיר",
    "ישוב",
    "יישוב",
    "רשות",
    "municipality",
    "street",
    "רחוב",
    "address",
    "כתובת",
    "status",
    "סטטוס",
    "מצב",
    "שלב",
    "יחידות",
    "יח\"ד",
    "דירות",
    "geometry",
    "geom",
    "shape",
    "lat",
    "lon",
    "geo",
]

RELEVANCE_KEYWORDS = STRICT_URBAN_RENEWAL_KEYWORDS + BROAD_RELATED_KEYWORDS
PREFERRED_FORMATS = {"CSV", "XLSX", "XLS", "JSON", "GEOJSON", "ZIP"}

MANUAL_KNOWN_RESOURCE_IDS = [
    {
        "source_name": "data.gov.il - מתחמי התחדשות עירונית",
        "resource_id": "f65a0daf-f737-49c5-9424-d378d52104f5",
        "source_url": "https://data.gov.il/he/datasets/ministry_of_housing/urban_renewal/f65a0daf-f737-49c5-9424-d378d52104f5",
        "source_type": "government_open_data_resource",
        "priority": 1,
        "expected_format": "CKAN DataStore",
        "access_method": "PUBLIC_API",
        "notes": "Official urban renewal complexes from Ministry of Housing / data.gov.il",
    },
    {
        "source_name": "data.gov.il - מתחמי התחדשות עירונית מוכרזים GIS",
        "resource_id": "ceb7bbb0-e2db-4e87-8a6c-0a250f5de001",
        "source_url": "https://data.gov.il/he/datasets/ministry_of_housing/gis_urban_renewal/ceb7bbb0-e2db-4e87-8a6c-0a250f5de001",
        "source_type": "government_open_data_gis_resource",
        "priority": 1,
        "expected_format": "CKAN DataStore",
        "access_method": "PUBLIC_API",
        "notes": "Declared urban renewal complexes GIS layer",
    },
    {
        "source_name": "data.gov.il - תוכניות מתאר של התחדשות עירונית",
        "resource_id": "38555a84-8523-4ab1-9fe5-df6b523c15ea",
        "source_url": "https://data.gov.il/he/datasets/ministry_of_housing/gis_urban_renewal/38555a84-8523-4ab1-9fe5-df6b523c15ea",
        "source_type": "government_open_data_gis_resource",
        "priority": 1,
        "expected_format": "CKAN DataStore",
        "access_method": "PUBLIC_API",
        "notes": "Urban renewal master plans GIS layer",
    },
]

# Optional future fallback: paste verified direct public CSV/API resource URLs here after manual discovery.
# Keep this empty until real URLs are identified; do not add examples or invented sources.
MANUAL_KNOWN_RESOURCE_URLS: List[Dict[str, Any]] = []

SOURCES = [
    {
        "source_name": "data.gov.il CKAN package_search",
        "source_url": DATA_GOV_CKAN_ENDPOINT,
        "source_type": "government_open_data_api",
        "priority": 1,
        "expected_format": "JSON",
        "access_method": "PUBLIC_API",
        "notes": "Dynamic CKAN search for urban-renewal-related public datasets.",
    },
    {
        "source_name": "data.gov.il cities streets and GIS discovery",
        "source_url": DATA_GOV_CKAN_ENDPOINT,
        "source_type": "government_open_data_api",
        "priority": 1,
        "expected_format": "CSV/XLSX/JSON/GEOJSON/ZIP",
        "access_method": "PUBLIC_API",
        "notes": "Official cities, streets, planning, and GIS datasets discovered through CKAN search.",
    },
    {
        "source_name": "GovMap public website",
        "source_url": "https://www.govmap.gov.il/",
        "source_type": "government_map_portal",
        "priority": 2,
        "expected_format": "HTML/GIS service",
        "access_method": "simple GET check only",
        "notes": "Urban renewal layers may exist, but direct stable downloads are not assumed here.",
    },
    {
        "source_name": "Planning Administration public page",
        "source_url": "https://www.gov.il/he/departments/planning_administration/govil-landing-page",
        "source_type": "government_planning_portal",
        "priority": 2,
        "expected_format": "HTML/API references",
        "access_method": "simple GET check only",
        "notes": "XPLAN/GIS access should be investigated later without scraping fragile viewers.",
    },
    {
        "source_name": "Mavat planning information public page",
        "source_url": "https://mavat.iplan.gov.il/SV4/1/99005041619/310",
        "source_type": "planning_information_portal",
        "priority": 2,
        "expected_format": "HTML",
        "access_method": "simple GET check only",
        "notes": "Document reachability only; no deep scraping or automation in Notebook 01.",
    },
    {
        "source_name": "Tel Aviv Open Data portal",
        "source_url": "https://opendata.tel-aviv.gov.il/",
        "source_type": "municipal_open_data_portal",
        "priority": 3,
        "expected_format": "HTML/API later",
        "access_method": "simple GET check only",
        "notes": "Later pilot source; avoid municipal scraping in this first notebook.",
    },
    {
        "source_name": "Municipal urban renewal pages",
        "source_url": "manual_later",
        "source_type": "municipal_web_pages",
        "priority": 3,
        "expected_format": "HTML",
        "access_method": "manual later",
        "notes": "Intentionally not scraped in Notebook 01.",
    },
    {
        "source_name": "Municipal permit systems",
        "source_url": "manual_later",
        "source_type": "municipal_permit_systems",
        "priority": 3,
        "expected_format": "HTML/API varies",
        "access_method": "manual/API investigation later",
        "notes": "Often login-based or unstable; intentionally not automated in Notebook 01.",
    },
]

source_inventory_records: List[Dict[str, Any]] = []
api_diagnostics_records: List[Dict[str, Any]] = []
loaded_datasets: List[Dict[str, Any]] = []
combined_frames: List[pd.DataFrame] = []
combined_df = pd.DataFrame()
final_check_df = pd.DataFrame()
known_resource_results: List[Dict[str, Any]] = []
browser_export_results: List[Dict[str, Any]] = []
browser_export_inspection_rows_extra: List[Dict[str, Any]] = []

pd.DataFrame(SOURCES)


,source_name,source_url,source_type,priority,expected_format,access_method,notes
0,data.gov.il CKAN package_search,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,PUBLIC_API,Dynamic CKAN search for urban-renewal-related ...
1,data.gov.il cities streets and GIS discovery,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,CSV/XLSX/JSON/GEOJSON/ZIP,PUBLIC_API,"Official cities, streets, planning, and GIS da..."
2,GovMap public website,https://www.govmap.gov.il/,government_map_portal,2,HTML/GIS service,simple GET check only,"Urban renewal layers may exist, but direct sta..."
3,Planning Administration public page,https://www.gov.il/he/departments/planning_adm...,government_planning_portal,2,HTML/API references,simple GET check only,XPLAN/GIS access should be investigated later ...
4,Mavat planning information public page,https://mavat.iplan.gov.il/SV4/1/99005041619/310,planning_information_portal,2,HTML,simple GET check only,Document reachability only; no deep scraping o...
5,Tel Aviv Open Data portal,https://opendata.tel-aviv.gov.il/,municipal_open_data_portal,3,HTML/API later,simple GET check only,Later pilot source; avoid municipal scraping i...
6,Municipal urban renewal pages,manual_later,municipal_web_pages,3,HTML,manual later,Intentionally not scraped in Notebook 01.
7,Municipal permit systems,manual_later,municipal_permit_systems,3,HTML/API varies,manual/API investigation later,Often login-based or unstable; intentionally n...


## 4 - Search data.gov.il CKAN for Relevant Datasets

Search the CKAN API dynamically using Hebrew and English search terms. The output captures all discovered resources and then filters likely relevant resources using metadata keywords and preferred file formats.

In [44]:
search_rows: List[Dict[str, Any]] = []


def parse_ckan_json_result(result: Dict[str, Any]) -> tuple[Optional[dict], bool, Optional[str]]:
    """Parse a CKAN API result and require payload['success'] == True."""
    response = result.get("response_object")
    if response is None:
        return None, False, result.get("error_message") or "No response object."
    try:
        payload = response.json()
    except Exception as exc:
        return None, False, f"JSON parse failed: {type(exc).__name__}: {exc}"
    if payload.get("success") is not True:
        return payload, False, f"CKAN API returned success={payload.get('success')}: {payload.get('error')}"
    return payload, True, None


def run_ckan_package_search(search_term: str) -> tuple[Optional[dict], Dict[str, Any], Optional[str]]:
    """Try CKAN package_search with GET, then POST JSON if GET is blocked with 403."""
    get_result = safe_request(
        DATA_GOV_CKAN_ENDPOINT,
        method="GET",
        params={"q": search_term, "rows": 100},
        timeout=REQUEST_TIMEOUT,
    )
    get_payload, get_parsed_success, get_error = parse_ckan_json_result(get_result)
    add_api_diagnostic(
        api_diagnostics_records,
        endpoint=DATA_GOV_CKAN_ENDPOINT,
        method="GET",
        search_term=search_term,
        response_result=get_result,
        parsed_json_success=get_parsed_success,
        error_message=get_error or get_result.get("error_message"),
    )
    if get_parsed_success:
        return get_payload, get_result, None

    if get_result.get("status_code") == 403:
        post_result = safe_request(
            DATA_GOV_CKAN_ENDPOINT,
            method="POST",
            json_payload={"q": search_term, "rows": 100},
            timeout=REQUEST_TIMEOUT,
        )
        post_payload, post_parsed_success, post_error = parse_ckan_json_result(post_result)
        add_api_diagnostic(
            api_diagnostics_records,
            endpoint=DATA_GOV_CKAN_ENDPOINT,
            method="POST",
            search_term=search_term,
            response_result=post_result,
            parsed_json_success=post_parsed_success,
            error_message=post_error or post_result.get("error_message"),
        )
        if post_parsed_success:
            return post_payload, post_result, None
        if post_result.get("status_code") == 403:
            return None, post_result, "HTTP 403 - likely blocked from this environment or requires different access path"
        return None, post_result, post_error or post_result.get("error_message")

    return None, get_result, get_error or get_result.get("error_message")


if RUN_DATA_GOV_SEARCH:
    for search_term in SEARCH_TERMS:
        print(f"Searching data.gov.il CKAN: {search_term}")
        payload, result, search_error = run_ckan_package_search(search_term)
        if payload is None:
            add_inventory_record(
                source_inventory_records,
                source_name=f"data.gov.il search: {search_term}",
                source_url=DATA_GOV_CKAN_ENDPOINT,
                source_type="government_open_data_api",
                priority=1,
                expected_format="JSON",
                access_method="CKAN package_search GET then POST fallback",
                access_level="FAILED",
                download_status="failed",
                http_status_code=result.get("status_code"),
                content_type=result.get("content_type"),
                load_status="not_attempted",
                error_message=search_error,
                notes="CKAN search failed after legitimate GET/POST attempts; continuing with remaining searches.",
                load_method="ckan_package_search",
                content_validation_status="not_applicable",
            )
            continue

        packages = payload.get("result", {}).get("results", [])
        for package in packages:
            organization = package.get("organization") or {}
            resources = package.get("resources") or []
            if not resources:
                search_rows.append(
                    {
                        "search_term": search_term,
                        "package_title": package.get("title"),
                        "package_name": package.get("name"),
                        "organization": organization.get("title") or organization.get("name"),
                        "package_notes": package.get("notes"),
                        "metadata_created": package.get("metadata_created"),
                        "metadata_modified": package.get("metadata_modified"),
                        "resource_name": None,
                        "resource_url": None,
                        "resource_format": None,
                        "resource_id": None,
                    }
                )
            for resource in resources:
                search_rows.append(
                    {
                        "search_term": search_term,
                        "package_title": package.get("title"),
                        "package_name": package.get("name"),
                        "organization": organization.get("title") or organization.get("name"),
                        "package_notes": package.get("notes"),
                        "metadata_created": package.get("metadata_created"),
                        "metadata_modified": package.get("metadata_modified"),
                        "resource_name": resource.get("name"),
                        "resource_url": resource.get("url"),
                        "resource_format": resource.get("format"),
                        "resource_id": resource.get("id"),
                    }
                )
else:
    print("RUN_DATA_GOV_SEARCH is False; skipping CKAN search.")

SEARCH_RESULT_COLUMNS = [
    "search_term",
    "package_title",
    "package_name",
    "organization",
    "package_notes",
    "metadata_created",
    "metadata_modified",
    "resource_name",
    "resource_url",
    "resource_format",
    "resource_id",
]
search_results_df = pd.DataFrame(search_rows, columns=SEARCH_RESULT_COLUMNS)
search_results_path = METADATA_DIR / "data_gov_search_results.csv"
search_results_df.to_csv(search_results_path, index=False, encoding="utf-8-sig")
print(f"Saved CKAN search results: {search_results_path.relative_to(PROJECT_ROOT)} ({len(search_results_df):,} rows)")

api_diagnostics_df = save_api_diagnostics()

if not search_results_df.empty:
    metadata_text = search_results_df[
        ["package_title", "package_name", "organization", "package_notes", "resource_name", "resource_format"]
    ].fillna("").agg(" | ".join, axis=1)
    format_ok = search_results_df["resource_format"].fillna("").str.upper().isin(PREFERRED_FORMATS)
    url_ok = search_results_df["resource_url"].notna() & (search_results_df["resource_url"].astype(str).str.len() > 0)
    strict_matches = metadata_text.apply(lambda text: _matched_keywords(text, STRICT_URBAN_RENEWAL_KEYWORDS))
    broad_matches = metadata_text.apply(lambda text: _matched_keywords(text, BROAD_RELATED_KEYWORDS))
    explicit_source_matches = metadata_text.apply(lambda text: _matched_keywords(text, EXPLICIT_URBAN_RENEWAL_SOURCE_KEYWORDS))

    candidates_df = search_results_df[url_ok & ((strict_matches.apply(bool)) | (broad_matches.apply(bool)) | (explicit_source_matches.apply(bool)))].copy()
    candidates_df["strict_urban_renewal_match"] = strict_matches.loc[candidates_df.index].apply(bool).values
    candidates_df["broad_related_match"] = broad_matches.loc[candidates_df.index].apply(bool).values
    candidates_df["explicit_urban_renewal_source"] = explicit_source_matches.loc[candidates_df.index].apply(bool).values
    candidates_df["matched_strict_keywords"] = strict_matches.loc[candidates_df.index].apply(lambda hits: json.dumps(hits, ensure_ascii=False)).values
    candidates_df["matched_broad_keywords"] = broad_matches.loc[candidates_df.index].apply(lambda hits: json.dumps(hits, ensure_ascii=False)).values
    candidates_df["detected_relevance_reason"] = metadata_text.loc[candidates_df.index].apply(lambda text: detect_relevance_reason(text, RELEVANCE_KEYWORDS)).values
    candidates_df["preferred_format"] = format_ok.loc[candidates_df.index].values

    def decide_candidate(row: pd.Series) -> pd.Series:
        if not bool(row["preferred_format"]):
            return pd.Series({"candidate_decision": "exclude", "candidate_decision_reason": "Relevant metadata but unsupported or missing resource format."})
        if bool(row["strict_urban_renewal_match"]) or bool(row["explicit_urban_renewal_source"]):
            return pd.Series({"candidate_decision": "include_for_download", "candidate_decision_reason": "Strict urban-renewal metadata/source match."})
        if bool(row["broad_related_match"]):
            return pd.Series({"candidate_decision": "metadata_only", "candidate_decision_reason": "Broad planning/housing metadata only; not strict enough for automatic ingestion."})
        return pd.Series({"candidate_decision": "exclude", "candidate_decision_reason": "No relevant metadata match."})

    candidates_df = pd.concat([candidates_df, candidates_df.apply(decide_candidate, axis=1)], axis=1)
    candidates_df = candidates_df.drop_duplicates(subset=["resource_id", "resource_url", "candidate_decision"]).reset_index(drop=True)
else:
    candidates_df = pd.DataFrame(columns=list(search_results_df.columns) + [
        "strict_urban_renewal_match",
        "broad_related_match",
        "explicit_urban_renewal_source",
        "matched_strict_keywords",
        "matched_broad_keywords",
        "detected_relevance_reason",
        "preferred_format",
        "candidate_decision",
        "candidate_decision_reason",
    ])

candidate_resources_path = METADATA_DIR / "data_gov_candidate_resources.csv"
candidates_df.to_csv(candidate_resources_path, index=False, encoding="utf-8-sig")
print(f"Saved candidate resources: {candidate_resources_path.relative_to(PROJECT_ROOT)} ({len(candidates_df):,} rows)")
print(candidates_df["candidate_decision"].value_counts(dropna=False).to_string() if not candidates_df.empty else "No candidates found.")
candidates_df.head(20)


Searching data.gov.il CKAN: התחדשות עירונית
Searching data.gov.il CKAN: פינוי בינוי
Searching data.gov.il CKAN: תמ"א 38
Searching data.gov.il CKAN: מתחמי התחדשות
Searching data.gov.il CKAN: רשות ממשלתית להתחדשות עירונית
Searching data.gov.il CKAN: urban renewal
Searching data.gov.il CKAN: renewal
Searching data.gov.il CKAN: housing
Searching data.gov.il CKAN: תכנון
Searching data.gov.il CKAN: תכנית
Searching data.gov.il CKAN: תוכניות
Saved CKAN search results: data\metadata\data_gov_search_results.csv (0 rows)
Saved API diagnostics: data\metadata\api_diagnostics.csv (22 rows)
Saved candidate resources: data\metadata\data_gov_candidate_resources.csv (0 rows)
No candidates found.


,search_term,package_title,package_name,organization,package_notes,metadata_created,metadata_modified,resource_name,resource_url,resource_format,resource_id,strict_urban_renewal_match,broad_related_match,explicit_urban_renewal_source,matched_strict_keywords,matched_broad_keywords,detected_relevance_reason,preferred_format,candidate_decision,candidate_decision_reason


## 5 - Download Candidate Resources

Always load the confirmed official data.gov.il urban-renewal resource IDs through paginated CKAN DataStore calls, even if `package_search` fails. Then process strict resources discovered dynamically through `package_search`, and keep broad-only candidates as metadata. The optional `MANUAL_KNOWN_RESOURCE_URLS` list stays empty by default; later verified direct public URLs can be added there and will use the same validation and quality gates.


In [45]:
def load_ckan_datastore_all_records(
    resource_id: str,
    page_size: int = 5_000,
    max_records: Optional[int] = None,
) -> Dict[str, Any]:
    """Load all available records from CKAN DataStore with robust pagination."""
    records: List[Dict[str, Any]] = []
    api_attempts_metadata: List[Dict[str, Any]] = []
    total_records_reported: Optional[int] = None
    offset = 0
    error_message = None

    while True:
        limit = page_size
        if max_records is not None:
            remaining = max_records - len(records)
            if remaining <= 0:
                break
            limit = min(page_size, remaining)

        params = {"resource_id": resource_id, "limit": limit, "offset": offset}
        request_result = safe_request(DATA_GOV_DATASTORE_ENDPOINT, params=params, timeout=REQUEST_TIMEOUT)
        attempt = {
            "endpoint": DATA_GOV_DATASTORE_ENDPOINT,
            "method": "GET",
            "resource_id": resource_id,
            "limit": limit,
            "offset": offset,
            "status_code": request_result.get("status_code"),
            "content_type": request_result.get("content_type"),
            "parsed_json_success": False,
            "records_returned": 0,
            "error_message": request_result.get("error_message"),
        }

        parsed_json_success = False
        diagnostic_error = request_result.get("error_message")
        response = request_result.get("response_object")
        payload = None

        if response is not None:
            validation = validate_response_content(response, expected_format="JSON", source_url=DATA_GOV_DATASTORE_ENDPOINT)
            if not validation["is_valid_download"]:
                diagnostic_error = validation["validation_error"]
                attempt["error_message"] = diagnostic_error
                api_attempts_metadata.append(attempt)
                add_api_diagnostic(
                    api_diagnostics_records,
                    endpoint=DATA_GOV_DATASTORE_ENDPOINT,
                    method="GET",
                    search_term=f"resource_id:{resource_id};offset:{offset}",
                    response_result=request_result,
                    parsed_json_success=False,
                    error_message=diagnostic_error,
                )
                error_message = diagnostic_error
                break

            try:
                payload = response.json()
                parsed_json_success = payload.get("success") is True
                if not parsed_json_success:
                    diagnostic_error = f"CKAN datastore_search returned success={payload.get('success')}: {payload.get('error')}"
            except Exception as exc:
                diagnostic_error = f"JSON parse failed: {type(exc).__name__}: {exc}"

        attempt["parsed_json_success"] = parsed_json_success
        attempt["error_message"] = diagnostic_error

        if not request_result.get("success") or not parsed_json_success or payload is None:
            api_attempts_metadata.append(attempt)
            add_api_diagnostic(
                api_diagnostics_records,
                endpoint=DATA_GOV_DATASTORE_ENDPOINT,
                method="GET",
                search_term=f"resource_id:{resource_id};offset:{offset}",
                response_result=request_result,
                parsed_json_success=parsed_json_success,
                error_message=diagnostic_error,
            )
            error_message = diagnostic_error or "CKAN datastore_search request failed."
            break

        result_payload = payload.get("result", {})
        page_records = result_payload.get("records", [])
        if total_records_reported is None:
            total_records_reported = result_payload.get("total")
        if not isinstance(page_records, list):
            error_message = "CKAN datastore_search result.records was not a list."
            attempt["error_message"] = error_message
            api_attempts_metadata.append(attempt)
            add_api_diagnostic(
                api_diagnostics_records,
                endpoint=DATA_GOV_DATASTORE_ENDPOINT,
                method="GET",
                search_term=f"resource_id:{resource_id};offset:{offset}",
                response_result=request_result,
                parsed_json_success=False,
                error_message=error_message,
            )
            break

        attempt["records_returned"] = len(page_records)
        api_attempts_metadata.append(attempt)
        add_api_diagnostic(
            api_diagnostics_records,
            endpoint=DATA_GOV_DATASTORE_ENDPOINT,
            method="GET",
            search_term=f"resource_id:{resource_id};offset:{offset}",
            response_result=request_result,
            parsed_json_success=True,
            error_message=None,
        )

        if not page_records:
            break

        records.extend(page_records)
        offset += len(page_records)

        if total_records_reported is not None and offset >= int(total_records_reported):
            break
        if len(page_records) < limit:
            break

    df = pd.DataFrame.from_records(records) if records else pd.DataFrame()
    success = bool(records) and error_message is None
    if not records and error_message is None:
        error_message = "CKAN datastore_search returned no records."
    return {
        "success": success,
        "dataframe": df,
        "records": records,
        "total_records_reported": total_records_reported,
        "total_records_loaded": len(records),
        "error_message": error_message,
        "api_attempts_metadata": api_attempts_metadata,
    }


def save_known_resource_raw_outputs(resource_id: str, records: List[Dict[str, Any]], df: pd.DataFrame) -> Dict[str, Path]:
    """Save raw DataStore records as JSON and CSV for reproducibility."""
    json_path = RAW_DIR / f"data_gov_{sanitize_filename(resource_id)}_records.json"
    csv_path = RAW_DIR / f"data_gov_{sanitize_filename(resource_id)}_records.csv"
    json_path.write_text(json.dumps(records, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    return {"json_path": json_path, "csv_path": csv_path}


def load_from_ckan_datastore(resource_id: str, source_name: str, resource_name: str) -> Dict[str, Any]:
    """Try CKAN DataStore API for dynamically discovered resources."""
    result = {
        "success": False,
        "df": None,
        "file_path": None,
        "http_status_code": None,
        "content_type": None,
        "error_message": None,
        "content_validation_status": None,
        "content_validation_error": None,
        "detected_content_kind": "json",
        "total_records_reported": None,
        "total_records_loaded": 0,
    }
    if not resource_id or str(resource_id).lower() == "nan":
        result["error_message"] = "Missing resource_id."
        return result

    datastore_result = load_ckan_datastore_all_records(resource_id, page_size=5_000, max_records=50_000)
    attempts = datastore_result.get("api_attempts_metadata") or []
    if attempts:
        last_attempt = attempts[-1]
        result["http_status_code"] = last_attempt.get("status_code")
        result["content_type"] = last_attempt.get("content_type")
    result["total_records_reported"] = datastore_result.get("total_records_reported")
    result["total_records_loaded"] = datastore_result.get("total_records_loaded")

    if datastore_result["success"]:
        df = datastore_result["dataframe"]
        paths = save_known_resource_raw_outputs(resource_id, datastore_result["records"], df)
        result.update(
            {
                "success": True,
                "df": df,
                "file_path": paths["csv_path"],
                "error_message": None,
                "content_validation_status": "valid",
                "content_validation_error": None,
            }
        )
    else:
        result["error_message"] = datastore_result.get("error_message")
        result["content_validation_status"] = "invalid" if result["error_message"] else "not_validated"
    return result


def evaluate_dataset_quality(
    df: Optional[pd.DataFrame],
    strict_urban_renewal_match: bool,
    explicit_urban_renewal_source: bool,
) -> Dict[str, Any]:
    """Apply quality gates before a loaded dataset may enter the raw collected CSV."""
    if df is None:
        return {"is_valid_dataset": False, "is_relevant_dataset": False, "final_included_in_raw_csv": False, "reason": "No dataframe was loaded."}
    if df.empty:
        return {"is_valid_dataset": False, "is_relevant_dataset": False, "final_included_in_raw_csv": False, "reason": "DataFrame is empty."}
    if dataframe_looks_like_html(df):
        return {"is_valid_dataset": False, "is_relevant_dataset": False, "final_included_in_raw_csv": False, "reason": "DataFrame appears to be HTML/security content parsed as data."}
    if df.shape[1] < 2:
        return {"is_valid_dataset": False, "is_relevant_dataset": False, "final_included_in_raw_csv": False, "reason": "DataFrame has fewer than 2 columns."}

    has_relevant_cols = dataset_has_relevant_columns(df.columns)
    strict_or_explicit = bool(strict_urban_renewal_match or explicit_urban_renewal_source)
    metadata_or_column_relevant = strict_or_explicit or has_relevant_cols
    is_valid_dataset = bool(metadata_or_column_relevant)
    is_relevant_dataset = bool(strict_or_explicit and metadata_or_column_relevant)
    final_included = bool(is_valid_dataset and is_relevant_dataset)

    if not is_valid_dataset:
        reason = "No strict metadata match and no relevant planning/location/status columns detected."
    elif not is_relevant_dataset:
        reason = "Dataset has broad/relevant-looking columns but lacks strict urban-renewal metadata/source match."
    else:
        reason = "Passed strict urban-renewal relevance and dataset quality gates."
    return {
        "is_valid_dataset": is_valid_dataset,
        "is_relevant_dataset": is_relevant_dataset,
        "final_included_in_raw_csv": final_included,
        "reason": reason,
    }


known_resource_results = []
print(f"Known data.gov.il urban-renewal resource IDs to attempt: {len(MANUAL_KNOWN_RESOURCE_IDS):,}")
for known_source in MANUAL_KNOWN_RESOURCE_IDS:
    resource_id = known_source["resource_id"]
    source_name = known_source["source_name"]
    source_url = known_source["source_url"]
    source_type = known_source["source_type"]
    print(f"Loading known official DataStore resource: {source_name} ({resource_id})")

    known_result = load_ckan_datastore_all_records(resource_id, page_size=5_000, max_records=None)
    df = known_result["dataframe"]
    paths = {"json_path": None, "csv_path": None}
    inspection = {"num_rows": None, "num_columns": None, "columns": []}
    is_valid_dataset = False
    is_relevant_dataset = False
    final_included = False
    quality_reason = known_result.get("error_message") or "Not evaluated."

    if known_result["success"] and df is not None and not df.empty:
        paths = save_known_resource_raw_outputs(resource_id, known_result["records"], df)
        inspection = inspect_dataframe(df)
        is_valid_dataset = bool(not df.empty and not dataframe_looks_like_html(df) and df.shape[1] >= 2)
        is_relevant_dataset = True
        final_included = bool(is_valid_dataset and is_relevant_dataset)
        quality_reason = "Known official data.gov.il urban renewal resource."

        loaded_datasets.append(
            {
                "source_name": source_name,
                "source_url": source_url,
                "source_type": source_type,
                "file_path": paths["csv_path"],
                "df": df,
                "inspection": inspection,
                "detected_relevance_reason": "Known official data.gov.il urban renewal resource",
                "load_method": "ckan_datastore_search_paginated",
                "final_included_in_raw_csv": final_included,
            }
        )

        if final_included:
            df_for_collect = df.copy()
            df_for_collect["source_name"] = source_name
            df_for_collect["source_url"] = source_url
            df_for_collect["source_type"] = source_type
            df_for_collect["original_resource_id"] = resource_id
            df_for_collect["original_file_path"] = str(paths["csv_path"].relative_to(PROJECT_ROOT))
            df_for_collect["original_row_index"] = np.arange(len(df_for_collect))
            df_for_collect["ingestion_timestamp"] = LAST_CHECKED
            df_for_collect["detected_relevance_reason"] = "Known official data.gov.il urban renewal resource"
            combined_frames.append(df_for_collect)

    attempts = known_result.get("api_attempts_metadata") or []
    last_attempt = attempts[-1] if attempts else {}
    known_resource_results.append(
        {
            "source_name": source_name,
            "resource_id": resource_id,
            "success": bool(final_included),
            "rows_loaded": int(known_result.get("total_records_loaded") or 0),
            "total_records_reported": known_result.get("total_records_reported"),
            "error_message": known_result.get("error_message"),
        }
    )

    add_inventory_record(
        source_inventory_records,
        source_name=source_name,
        source_url=source_url,
        source_type=source_type,
        priority=int(known_source.get("priority", 1)),
        expected_format=known_source.get("expected_format", "CKAN DataStore"),
        access_method="CKAN datastore_search paginated by known official resource_id",
        access_level="PUBLIC_API" if known_result["success"] else "FAILED",
        download_status="downloaded" if known_result["success"] else "failed",
        http_status_code=last_attempt.get("status_code"),
        content_type=last_attempt.get("content_type"),
        file_path=paths["csv_path"],
        load_status="loaded" if known_result["success"] else "failed",
        num_rows=inspection.get("num_rows"),
        num_columns=inspection.get("num_columns"),
        columns=inspection.get("columns"),
        error_message=known_result.get("error_message"),
        notes=f"{known_source.get('notes')}; {quality_reason}",
        load_method="ckan_datastore_search_paginated",
        content_validation_status="valid" if known_result["success"] else "invalid",
        content_validation_error=known_result.get("error_message"),
        detected_content_kind="json",
        is_valid_dataset=is_valid_dataset,
        is_relevant_dataset=is_relevant_dataset,
        strict_urban_renewal_match=True,
        matched_strict_keywords=["known official data.gov.il urban renewal resource"],
        final_included_in_raw_csv=final_included,
        original_resource_id=resource_id,
        total_records_reported=known_result.get("total_records_reported"),
        total_records_loaded=known_result.get("total_records_loaded"),
    )


download_candidates_df = candidates_df[candidates_df.get("candidate_decision", pd.Series(dtype=str)) == "include_for_download"].copy() if not candidates_df.empty else pd.DataFrame()

manual_seed_rows: List[Dict[str, Any]] = []
for manual_idx, manual_source in enumerate(MANUAL_KNOWN_RESOURCE_URLS):
    resource_url = str(manual_source.get("resource_url") or manual_source.get("source_url") or "").strip()
    if not resource_url:
        warnings.warn(f"Skipping manual known resource {manual_idx + 1}: missing resource_url/source_url")
        continue
    manual_seed_rows.append(
        {
            "search_term": "manual_known_resource_url",
            "package_title": manual_source.get("source_name") or f"manual known urban renewal resource {manual_idx + 1}",
            "package_name": manual_source.get("package_name") or "manual_known_resource_url",
            "organization": manual_source.get("organization"),
            "package_notes": manual_source.get("notes") or "Manually verified direct public resource URL.",
            "metadata_created": None,
            "metadata_modified": None,
            "resource_name": manual_source.get("resource_name") or manual_source.get("source_name") or f"manual_resource_{manual_idx + 1}",
            "resource_url": resource_url,
            "resource_format": manual_source.get("expected_format") or manual_source.get("resource_format") or "CSV",
            "resource_id": manual_source.get("resource_id") or "",
            "strict_urban_renewal_match": bool(manual_source.get("strict_urban_renewal_match", True)),
            "broad_related_match": bool(manual_source.get("broad_related_match", True)),
            "explicit_urban_renewal_source": bool(manual_source.get("explicit_urban_renewal_source", True)),
            "matched_strict_keywords": json.dumps(manual_source.get("matched_strict_keywords", ["manual_verified_urban_renewal_source"]), ensure_ascii=False),
            "matched_broad_keywords": json.dumps(manual_source.get("matched_broad_keywords", []), ensure_ascii=False),
            "detected_relevance_reason": manual_source.get("detected_relevance_reason") or "manual verified direct public urban-renewal resource",
            "preferred_format": True,
            "candidate_decision": "include_for_download",
            "candidate_decision_reason": "Manual verified direct resource URL; still subject to validation and quality gates.",
        }
    )

if manual_seed_rows:
    manual_candidates_df = pd.DataFrame(manual_seed_rows)
    download_candidates_df = pd.concat([download_candidates_df, manual_candidates_df], ignore_index=True, join="outer")
    print(f"Added {len(manual_candidates_df):,} manual known resource URL(s) to the validated download queue.")

metadata_only_candidates_df = candidates_df[candidates_df.get("candidate_decision", pd.Series(dtype=str)) != "include_for_download"].copy() if not candidates_df.empty else pd.DataFrame()
for _, row in metadata_only_candidates_df.iterrows():
    decision = str(row.get("candidate_decision") or "metadata_only")
    source_name = str(row.get("package_title") or row.get("package_name") or row.get("resource_name") or "data.gov.il metadata-only resource")
    matched_strict_keywords = json.loads(row.get("matched_strict_keywords") or "[]")
    add_inventory_record(
        source_inventory_records,
        source_name=source_name,
        source_url=str(row.get("resource_url") or ""),
        source_type="data.gov.il_ckan_resource_candidate",
        priority=1,
        expected_format=str(row.get("resource_format") or "unknown"),
        access_method="metadata discovery only; not downloaded automatically",
        access_level="NOT_RELEVANT" if decision == "exclude" else "UNKNOWN",
        download_status="not_attempted",
        load_status="not_attempted",
        error_message=None,
        notes=str(row.get("candidate_decision_reason") or "Candidate did not pass strict automatic-ingestion rules."),
        load_method="metadata_only",
        content_validation_status="not_applicable",
        is_valid_dataset=False,
        is_relevant_dataset=False,
        strict_urban_renewal_match=bool(row.get("strict_urban_renewal_match")),
        matched_strict_keywords=matched_strict_keywords,
        final_included_in_raw_csv=False,
    )

if RUN_DIRECT_DOWNLOADS and not download_candidates_df.empty:
    for idx, row in download_candidates_df.iterrows():
        resource_url = str(row.get("resource_url") or "").strip()
        resource_id = str(row.get("resource_id") or "").strip()
        source_name = str(row.get("package_title") or row.get("package_name") or row.get("resource_name") or f"data.gov.il resource {idx}")
        resource_name = str(row.get("resource_name") or "resource")
        expected_format = str(row.get("resource_format") or "unknown").upper()
        strict_match = bool(row.get("strict_urban_renewal_match"))
        explicit_source = bool(row.get("explicit_urban_renewal_source"))
        matched_strict_keywords = json.loads(row.get("matched_strict_keywords") or "[]")
        relevance_reason = str(row.get("detected_relevance_reason") or "strict metadata/source match")

        if resource_id and any(item["resource_id"] == resource_id for item in MANUAL_KNOWN_RESOURCE_IDS):
            print(f"Skipping dynamically discovered duplicate known resource: {source_name} ({resource_id})")
            continue

        print(f"Loading candidate {len(loaded_datasets) + 1}: {source_name} - {resource_name}")
        file_path = None
        df = None
        load_method = None
        download_status = "not_attempted"
        load_status = "not_attempted"
        http_status_code = None
        content_type = None
        validation_status = None
        validation_error = None
        detected_content_kind = None
        error_messages: List[str] = []
        inspection = {"num_rows": None, "num_columns": None, "columns": []}
        total_records_reported = None
        total_records_loaded = None

        datastore_result = load_from_ckan_datastore(resource_id, source_name, resource_name)
        http_status_code = datastore_result["http_status_code"]
        content_type = datastore_result["content_type"]
        validation_status = datastore_result["content_validation_status"]
        validation_error = datastore_result["content_validation_error"]
        detected_content_kind = datastore_result["detected_content_kind"]
        total_records_reported = datastore_result.get("total_records_reported")
        total_records_loaded = datastore_result.get("total_records_loaded")

        if datastore_result["success"]:
            df = datastore_result["df"]
            file_path = datastore_result["file_path"]
            load_method = "ckan_datastore_search_paginated"
            download_status = "downloaded"
            load_status = "loaded"
        else:
            if datastore_result.get("error_message"):
                error_messages.append(f"datastore_search: {datastore_result['error_message']}")
            print(f"  DataStore unavailable/no records; trying direct download. Reason: {datastore_result.get('error_message')}")

            request_result = safe_request(resource_url, timeout=REQUEST_TIMEOUT)
            http_status_code = request_result["status_code"]
            content_type = request_result["content_type"]
            if request_result["success"] and request_result["response_object"] is not None:
                validation = validate_response_content(request_result["response_object"], expected_format=expected_format, source_url=resource_url)
                validation_status = "valid" if validation["is_valid_download"] else "invalid"
                validation_error = validation["validation_error"]
                detected_content_kind = validation["detected_content_kind"]
                if validation["is_valid_download"]:
                    extension = infer_file_extension(resource_url, request_result["content_type"], fallback=f".{expected_format.lower()}" if expected_format else ".csv")
                    filename = f"data_gov_{sanitize_filename(source_name)}_{sanitize_filename(resource_name)}_{sanitize_filename(resource_id or idx)}{extension}"
                    file_path = RAW_DIR / filename
                    try:
                        save_response_content(request_result["response_object"], file_path)
                        download_status = "downloaded"
                        load_result = try_load_dataset(file_path)
                        if load_result["success"]:
                            df = load_result["df"]
                            load_method = "direct_download"
                            load_status = "loaded"
                            total_records_loaded = len(df)
                        else:
                            load_status = "failed"
                            error_messages.append(f"direct_download_load: {load_result['error_message']}")
                    except Exception as exc:
                        download_status = "failed"
                        load_status = "failed"
                        error_messages.append(f"direct_download_save/load: {type(exc).__name__}: {exc}")
                else:
                    download_status = "failed"
                    load_status = "failed"
                    error_messages.append(f"content_validation: {validation_error}")
            else:
                download_status = "failed"
                load_status = "failed"
                validation_status = "not_validated"
                error_messages.append(f"direct_download_request: {request_result.get('error_message')}")

        quality = evaluate_dataset_quality(df, strict_match, explicit_source)
        if df is not None and load_status == "loaded":
            inspection = inspect_dataframe(df)
            column_relevance = detect_relevance_reason(" | ".join(inspection["columns"]), DATASET_COLUMN_RELEVANCE_KEYWORDS)
            if column_relevance != "metadata/columns selected as relevant candidate":
                relevance_reason = f"{relevance_reason}; columns {column_relevance}"
            if not quality["final_included_in_raw_csv"]:
                error_messages.append(f"quality_gate: {quality['reason']}")

            loaded_datasets.append(
                {
                    "source_name": source_name,
                    "source_url": resource_url,
                    "source_type": "data.gov.il_ckan_resource",
                    "file_path": file_path,
                    "df": df,
                    "inspection": inspection,
                    "detected_relevance_reason": relevance_reason,
                    "load_method": load_method,
                    "final_included_in_raw_csv": quality["final_included_in_raw_csv"],
                }
            )

            if quality["final_included_in_raw_csv"]:
                df_for_collect = df.copy()
                df_for_collect["source_name"] = source_name
                df_for_collect["source_url"] = resource_url
                df_for_collect["source_type"] = "data.gov.il_ckan_resource"
                df_for_collect["original_resource_id"] = resource_id
                df_for_collect["original_file_path"] = str(file_path.relative_to(PROJECT_ROOT)) if file_path else None
                df_for_collect["original_row_index"] = np.arange(len(df_for_collect))
                df_for_collect["ingestion_timestamp"] = LAST_CHECKED
                df_for_collect["detected_relevance_reason"] = relevance_reason
                combined_frames.append(df_for_collect)

        access_level = "PUBLIC_API" if load_method and load_method.startswith("ckan_datastore") else classify_access_level(
            success=(download_status == "downloaded"),
            source_type="data.gov.il_ckan_resource",
            access_method="direct download from CKAN resource URL",
            download_status=download_status,
            notes=relevance_reason,
        )
        add_inventory_record(
            source_inventory_records,
            source_name=source_name,
            source_url=resource_url,
            source_type="data.gov.il_ckan_resource",
            priority=1,
            expected_format=expected_format,
            access_method="CKAN datastore_search first, then direct resource URL",
            access_level=access_level,
            download_status=download_status,
            http_status_code=http_status_code,
            content_type=content_type,
            file_path=file_path,
            load_status=load_status,
            num_rows=inspection.get("num_rows"),
            num_columns=inspection.get("num_columns"),
            columns=inspection.get("columns"),
            error_message="; ".join([msg for msg in error_messages if msg]),
            notes=f"{relevance_reason}; {quality['reason']}",
            load_method=load_method,
            content_validation_status=validation_status,
            content_validation_error=validation_error,
            detected_content_kind=detected_content_kind,
            is_valid_dataset=quality["is_valid_dataset"],
            is_relevant_dataset=quality["is_relevant_dataset"],
            strict_urban_renewal_match=strict_match,
            matched_strict_keywords=matched_strict_keywords,
            final_included_in_raw_csv=quality["final_included_in_raw_csv"],
            original_resource_id=resource_id,
            total_records_reported=total_records_reported,
            total_records_loaded=total_records_loaded,
        )
elif candidates_df.empty:
    print("No data.gov.il candidate resources were found for automatic download. Known resource ID ingestion was still attempted above.")
elif download_candidates_df.empty:
    print("No strict urban-renewal candidates qualified for automatic download. Broad-only candidates were kept as metadata only. Known resource ID ingestion was still attempted above.")
else:
    print("RUN_DIRECT_DOWNLOADS is False; skipping candidate downloads. Known resource ID ingestion was still attempted above.")

print(f"Loaded datasets: {len(loaded_datasets):,}")
print(f"Datasets included in raw CSV: {len(combined_frames):,}")
api_diagnostics_df = save_api_diagnostics()


Known data.gov.il urban-renewal resource IDs to attempt: 3
Loading known official DataStore resource: data.gov.il - מתחמי התחדשות עירונית (f65a0daf-f737-49c5-9424-d378d52104f5)
Loading known official DataStore resource: data.gov.il - מתחמי התחדשות עירונית מוכרזים GIS (ceb7bbb0-e2db-4e87-8a6c-0a250f5de001)
Loading known official DataStore resource: data.gov.il - תוכניות מתאר של התחדשות עירונית (38555a84-8523-4ab1-9fe5-df6b523c15ea)
No data.gov.il candidate resources were found for automatic download. Known resource ID ingestion was still attempted above.
Loaded datasets: 0
Datasets included in raw CSV: 0
Saved API diagnostics: data\metadata\api_diagnostics.csv (25 rows)


## Browser JSON Export Fallback

If Python `requests` receives a security or AccessDenied HTML page but the same official data.gov.il CKAN DataStore URL opens as JSON in a browser, save that browser response as a `.json` file under `data/manual_sources/`. This section ingests those local browser-saved JSON files using the same validation and quality gates.

In [46]:
browser_export_results = []
browser_export_inspection_rows_extra = []
known_official_resource_ids = {item["resource_id"] for item in MANUAL_KNOWN_RESOURCE_IDS}
browser_json_files = sorted(MANUAL_SOURCES_DIR.glob("*.json"))

print(f"Browser JSON export files found: {len(browser_json_files):,}")
if not browser_json_files:
    print(f"No browser-saved JSON files found in {MANUAL_SOURCES_DIR.relative_to(PROJECT_ROOT)}.")

for json_path in browser_json_files:
    source_name = json_path.stem
    source_url = "browser_saved_official_data_gov_json"
    source_type = "browser_export_official_data_gov_api"
    resource_id = None
    total_records_reported = None
    total_records_loaded = 0
    inspection = {"num_rows": None, "num_columns": None, "columns": []}
    error_message = None
    load_status = "failed"
    is_valid_dataset = False
    is_relevant_dataset = False
    final_included = False
    strict_match = False
    matched_strict_keywords: List[str] = []

    try:
        with json_path.open("r", encoding="utf-8-sig") as fh:
            payload = json.load(fh)

        if not isinstance(payload, dict):
            raise ValueError("Browser JSON export is not a JSON object.")
        if payload.get("success") is not True:
            raise ValueError(f"CKAN browser export success field is not True: {payload.get('success')}")

        result_payload = payload.get("result") or {}
        records = result_payload.get("records")
        resource_id = result_payload.get("resource_id") or payload.get("resource_id")
        total_records_reported = result_payload.get("total")

        if not isinstance(records, list) or not records:
            raise ValueError("CKAN browser export has no non-empty result.records list.")

        source_name = f"browser_export_{resource_id}" if resource_id else json_path.stem
        df = pd.DataFrame.from_records(records)
        total_records_loaded = int(len(df))
        strict_match = bool(resource_id in known_official_resource_ids)
        matched_strict_keywords = ["known official data.gov.il urban renewal resource"] if strict_match else []

        is_valid_dataset = bool(not df.empty and not dataframe_looks_like_html(df) and df.shape[1] >= 2)
        is_relevant_dataset = bool(strict_match)
        final_included = bool(is_valid_dataset and is_relevant_dataset)
        inspection = inspect_dataframe(df)
        load_status = "loaded"

        loaded_datasets.append(
            {
                "source_name": source_name,
                "source_url": source_url,
                "source_type": source_type,
                "file_path": json_path,
                "df": df,
                "inspection": inspection,
                "detected_relevance_reason": "Browser-saved official data.gov.il CKAN datastore_search JSON",
                "load_method": "browser_json_export",
                "final_included_in_raw_csv": final_included,
            }
        )

        if final_included:
            df_for_collect = df.copy()
            df_for_collect["source_name"] = source_name
            df_for_collect["source_url"] = source_url
            df_for_collect["source_type"] = source_type
            df_for_collect["original_resource_id"] = resource_id
            df_for_collect["original_file_path"] = str(json_path.relative_to(PROJECT_ROOT))
            df_for_collect["original_row_index"] = np.arange(len(df_for_collect))
            df_for_collect["ingestion_timestamp"] = LAST_CHECKED
            df_for_collect["detected_relevance_reason"] = "Browser-saved official data.gov.il CKAN datastore_search JSON"
            combined_frames.append(df_for_collect)
        elif not strict_match:
            error_message = "Browser JSON resource_id is not one of the known official urban-renewal resource IDs."
        elif not is_valid_dataset:
            error_message = "Browser JSON records did not pass dataset validity gates."

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        browser_export_inspection_rows_extra.append(
            {
                "source_name": source_name,
                "file_path": str(json_path.relative_to(PROJECT_ROOT)),
                "num_rows": 0,
                "num_columns": 0,
                "columns": json.dumps([], ensure_ascii=False),
                "missing_ratio_mean": None,
                "has_city_like_column": False,
                "has_street_like_column": False,
                "has_plan_like_column": False,
                "has_status_like_column": False,
                "has_units_like_column": False,
                "has_geometry_like_column": False,
            }
        )

    browser_export_results.append(
        {
            "file_path": str(json_path.relative_to(PROJECT_ROOT)),
            "resource_id": resource_id,
            "success": bool(final_included),
            "rows_loaded": total_records_loaded,
            "total_records_reported": total_records_reported,
            "error_message": error_message,
        }
    )

    add_inventory_record(
        source_inventory_records,
        source_name=source_name,
        source_url=source_url,
        source_type=source_type,
        priority=1,
        expected_format="Browser-saved CKAN DataStore JSON",
        access_method="local browser JSON export fallback",
        access_level="PUBLIC_API" if final_included else "FAILED",
        download_status="browser_saved_json",
        http_status_code=None,
        content_type="application/json",
        file_path=json_path,
        load_status=load_status,
        num_rows=inspection.get("num_rows"),
        num_columns=inspection.get("num_columns"),
        columns=inspection.get("columns"),
        error_message=error_message,
        notes="Browser-saved official data.gov.il CKAN datastore_search JSON",
        load_method="browser_json_export",
        content_validation_status="valid" if load_status == "loaded" else "invalid",
        content_validation_error=error_message,
        detected_content_kind="json",
        is_valid_dataset=is_valid_dataset,
        is_relevant_dataset=is_relevant_dataset,
        strict_urban_renewal_match=strict_match,
        matched_strict_keywords=matched_strict_keywords,
        final_included_in_raw_csv=final_included,
        original_resource_id=resource_id,
        total_records_reported=total_records_reported,
        total_records_loaded=total_records_loaded,
    )

browser_json_files_loaded_successfully = sum(1 for item in browser_export_results if item.get("success"))
browser_export_rows_loaded = sum(int(item.get("rows_loaded") or 0) for item in browser_export_results if item.get("success"))
print(f"Browser JSON files loaded successfully: {browser_json_files_loaded_successfully:,}")
print(f"Rows loaded from browser exports: {browser_export_rows_loaded:,}")
print(f"Datasets included in raw CSV after browser fallback: {len(combined_frames):,}")


Browser JSON export files found: 1
Browser JSON files loaded successfully: 1
Rows loaded from browser exports: 938
Datasets included in raw CSV after browser fallback: 1


## 6 - Optional Direct Source Checks

Check whether major public portals are reachable using simple GET requests only. This documents public access without scraping complex HTML, Power BI dashboards, or login-based systems.

In [47]:
if RUN_OPTIONAL_GIS_CHECKS:
    for source in SOURCES:
        source_url = source["source_url"]
        if source_url == "manual_later":
            add_inventory_record(
                source_inventory_records,
                source_name=source["source_name"],
                source_url=source_url,
                source_type=source["source_type"],
                priority=source["priority"],
                expected_format=source["expected_format"],
                access_method=source["access_method"],
                access_level="PUBLIC_MANUAL_ONLY",
                download_status="not_attempted",
                load_status="not_attempted",
                notes=source["notes"],
            )
            continue

        print(f"Checking reachability: {source['source_name']}")
        result = safe_request(source_url, timeout=REQUEST_TIMEOUT)
        access_level = classify_access_level(
            success=result["success"],
            source_type=source["source_type"],
            access_method=source["access_method"],
            download_status="reachable" if result["success"] else "failed",
            notes=source["notes"],
        )
        add_inventory_record(
            source_inventory_records,
            source_name=source["source_name"],
            source_url=source_url,
            source_type=source["source_type"],
            priority=source["priority"],
            expected_format=source["expected_format"],
            access_method=source["access_method"],
            access_level=access_level,
            download_status="reachable" if result["success"] else "failed",
            http_status_code=result["status_code"],
            content_type=result["content_type"],
            load_status="not_attempted",
            error_message=result["error_message"],
            notes=source["notes"],
        )
else:
    print("RUN_OPTIONAL_GIS_CHECKS is False; skipping direct source checks.")

Checking reachability: data.gov.il CKAN package_search
Checking reachability: data.gov.il cities streets and GIS discovery
Checking reachability: GovMap public website
Checking reachability: Planning Administration public page
Checking reachability: Mavat planning information public page
Checking reachability: Tel Aviv Open Data portal


## 7 - Build Source Inventory

The source inventory records every checked source/resource with access, download, loading, and inspection status.

In [48]:
source_inventory_df = pd.DataFrame(source_inventory_records)

required_inventory_columns = [
    "source_name",
    "source_url",
    "source_type",
    "priority",
    "expected_format",
    "access_method",
    "access_level",
    "download_status",
    "http_status_code",
    "content_type",
    "file_path",
    "load_status",
    "num_rows",
    "num_columns",
    "columns",
    "last_checked",
    "error_message",
    "notes",
    "load_method",
    "content_validation_status",
    "content_validation_error",
    "detected_content_kind",
    "is_valid_dataset",
    "is_relevant_dataset",
    "strict_urban_renewal_match",
    "matched_strict_keywords",
    "final_included_in_raw_csv",
    "original_resource_id",
    "total_records_reported",
    "total_records_loaded",
]
for col in required_inventory_columns:
    if col not in source_inventory_df.columns:
        source_inventory_df[col] = None
source_inventory_df = source_inventory_df[required_inventory_columns]

source_inventory_csv_path = METADATA_DIR / "source_inventory.csv"
source_inventory_json_path = METADATA_DIR / "source_inventory.json"
source_inventory_df.to_csv(source_inventory_csv_path, index=False, encoding="utf-8-sig")
source_inventory_df.to_json(source_inventory_json_path, orient="records", force_ascii=False, indent=2)

summary = {
    "total_sources_checked": int(len(source_inventory_df)),
    "successful_downloads": int((source_inventory_df["download_status"] == "downloaded").sum()) if not source_inventory_df.empty else 0,
    "successfully_loaded_datasets": int((source_inventory_df["load_status"] == "loaded").sum()) if not source_inventory_df.empty else 0,
    "failed_downloads": int((source_inventory_df["download_status"] == "failed").sum()) if not source_inventory_df.empty else 0,
    "failed_loads": int((source_inventory_df["load_status"] == "failed").sum()) if not source_inventory_df.empty else 0,
    "PUBLIC_DOWNLOAD_sources": int((source_inventory_df["access_level"] == "PUBLIC_DOWNLOAD").sum()) if not source_inventory_df.empty else 0,
    "PUBLIC_API_sources": int((source_inventory_df["access_level"] == "PUBLIC_API").sum()) if not source_inventory_df.empty else 0,
    "PUBLIC_BUT_FRAGILE_sources": int((source_inventory_df["access_level"] == "PUBLIC_BUT_FRAGILE").sum()) if not source_inventory_df.empty else 0,
    "invalid_HTML_or_security_pages_rejected": int(source_inventory_df["detected_content_kind"].isin(["html", "blocked_or_security_page"]).sum()) if not source_inventory_df.empty else 0,
    "final_included_in_raw_csv": int(source_inventory_df["final_included_in_raw_csv"].fillna(False).astype(bool).sum()) if not source_inventory_df.empty else 0,
}

print(f"Saved source inventory CSV: {source_inventory_csv_path.relative_to(PROJECT_ROOT)}")
print(f"Saved source inventory JSON: {source_inventory_json_path.relative_to(PROJECT_ROOT)}")
print(json.dumps(summary, ensure_ascii=False, indent=2))
source_inventory_df.head(20)


Saved source inventory CSV: data\metadata\source_inventory.csv
Saved source inventory JSON: data\metadata\source_inventory.json
{
  "total_sources_checked": 23,
  "successful_downloads": 0,
  "successfully_loaded_datasets": 1,
  "failed_downloads": 17,
  "failed_loads": 3,
  "PUBLIC_DOWNLOAD_sources": 1,
  "PUBLIC_API_sources": 3,
  "PUBLIC_BUT_FRAGILE_sources": 0,
  "invalid_HTML_or_security_pages_rejected": 0,
  "final_included_in_raw_csv": 1
}


,source_name,source_url,source_type,priority,expected_format,access_method,access_level,download_status,http_status_code,content_type,...,content_validation_error,detected_content_kind,is_valid_dataset,is_relevant_dataset,strict_urban_renewal_match,matched_strict_keywords,final_included_in_raw_csv,original_resource_id,total_records_reported,total_records_loaded
0,data.gov.il search: התחדשות עירונית,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN
1,data.gov.il search: פינוי בינוי,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN
2,"data.gov.il search: תמ""א 38",https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN
3,data.gov.il search: מתחמי התחדשות,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN
4,data.gov.il search: רשות ממשלתית להתחדשות עירונית,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN
5,data.gov.il search: urban renewal,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN
6,data.gov.il search: renewal,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN
7,data.gov.il search: housing,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN
8,data.gov.il search: תכנון,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN
9,data.gov.il search: תכנית,https://data.gov.il/api/3/action/package_search,government_open_data_api,1,JSON,CKAN package_search GET then POST fallback,FAILED,failed,403.0,text/html,...,None,None,None,None,None,[],False,None,NaN,NaN


## 8 - Dataset Inspection Report

Inspect each successfully loaded dataset and write a compact metadata summary with detected planning, address, unit, and geometry-like columns.

In [49]:
CITY_KEYWORDS = ["city", "עיר", "ישוב", "יישוב", "רשות", "municipality"]
STREET_KEYWORDS = ["street", "רחוב", "address", "כתובת"]
PLAN_KEYWORDS = ["plan", "תכנית", "תוכנית", "מספר תכנית", "מס' תכנית"]
STATUS_KEYWORDS = ["status", "סטטוס", "מצב", "שלב"]
UNITS_KEYWORDS = ["units", "יחידות", "יח\"ד", "דירות"]
GEOMETRY_KEYWORDS = ["geometry", "geom", "shape", "lat", "lon", "x", "y", "geo"]

inspection_rows: List[Dict[str, Any]] = []

for dataset in loaded_datasets:
    source_name = dataset["source_name"]
    file_path = dataset["file_path"]
    df = dataset["df"]
    inspection = dataset["inspection"]
    columns = inspection["columns"]

    print("=" * 100)
    print(f"Source: {source_name}")
    print(f"File: {file_path.relative_to(PROJECT_ROOT)}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {columns}")
    display(df.head(5))
    if df.shape[1]:
        display((df.isna().mean() * 100).sort_values(ascending=False).rename("missing_percent").to_frame())

    inspection_rows.append(
        {
            "source_name": source_name,
            "file_path": str(file_path.relative_to(PROJECT_ROOT)),
            "num_rows": inspection["num_rows"],
            "num_columns": inspection["num_columns"],
            "columns": json.dumps(columns, ensure_ascii=False),
            "missing_ratio_mean": inspection["missing_ratio_mean"],
            "has_city_like_column": has_keyword_in_columns(columns, CITY_KEYWORDS),
            "has_street_like_column": has_keyword_in_columns(columns, STREET_KEYWORDS),
            "has_plan_like_column": has_keyword_in_columns(columns, PLAN_KEYWORDS),
            "has_status_like_column": has_keyword_in_columns(columns, STATUS_KEYWORDS),
            "has_units_like_column": has_keyword_in_columns(columns, UNITS_KEYWORDS),
            "has_geometry_like_column": has_keyword_in_columns(columns, GEOMETRY_KEYWORDS),
        }
    )

dataset_inspection_summary_df = pd.DataFrame(inspection_rows)
dataset_inspection_summary_path = METADATA_DIR / "dataset_inspection_summary.csv"
dataset_inspection_summary_df.to_csv(dataset_inspection_summary_path, index=False, encoding="utf-8-sig")
print(f"Saved dataset inspection summary: {dataset_inspection_summary_path.relative_to(PROJECT_ROOT)} ({len(dataset_inspection_summary_df):,} rows)")
dataset_inspection_summary_df

Source: browser_export_f65a0daf-f737-49c5-9424-d378d52104f5
File: data\manual_sources\data_gov_f65a0daf_records.json
Shape: (938, 17)
Columns: ['_id', 'MisparMitham', 'Yeshuv', 'SemelYeshuv', 'ShemMitcham', 'YachadKayam', 'YachadTosafti', 'YachadMutza', 'TaarichHachraza', 'MisparTochnit', 'KishurLatar', 'SachHeterim', 'KishurLaMapa', 'Maslul', 'ShnatMatanTokef', 'Bebitzua', 'Status']


,_id,MisparMitham,Yeshuv,SemelYeshuv,ShemMitcham,YachadKayam,YachadTosafti,YachadMutza,TaarichHachraza,MisparTochnit,KishurLatar,SachHeterim,KishurLaMapa,Maslul,ShnatMatanTokef,Bebitzua,Status
0,1,4001,גבעתים ...,6300,ערבי נחל ...,126,108,530.0,20/08/2006,גב/490 ...,https://mavat.iplan.gov.il/SV4/1/5073314/310 ...,530,https://www.govmap.gov.il/map.html?lay=ADD_PRO...,מיסוי ...,2012 ...,לא ...,תכנית מאושרת - אחרי רישוי ...
1,2,4005,קרית אונו ...,2620,ישעיהו ...,198,198,396.0,20/08/2006,תממ/284 ...,https://mavat.iplan.gov.il/SV4/1/5048159/310 ...,228,https://www.govmap.gov.il/map.html?lay=ADD_PRO...,מיסוי ...,1998 ...,כן ...,תכנית מאושרת במימוש ...
2,3,4006,קרית אונו ...,2620,שאול המלך ...,180,48,531.0,27/05/2004,קא/מק/61/285/א ...,https://mavat.iplan.gov.il/SV4/1/5052333/310 ...,531,https://www.govmap.gov.il/map.html?lay=ADD_PRO...,מיסוי ...,2003 ...,לא ...,תכנית מאושרת - אחרי רישוי ...
3,4,4010,ראשון לציון ...,8300,רמת אליהו (פוזננסקי) ...,58,232,290.0,29/06/2017,413-0292680 ...,https://mavat.iplan.gov.il/SV4/1/4000346997/31...,116,https://www.govmap.gov.il/map.html?lay=ADD_PRO...,מיסוי ...,2014 ...,כן ...,תכנית מאושרת במימוש ...
4,5,4011,ראשון לציון ...,8300,סלע ...,283,0,1386.0,09/02/2012,רצ/מק/1/13/19/4 ...,https://mavat.iplan.gov.il/SV4/1/4097824/310 ...,676,https://www.govmap.gov.il/map.html?lay=ADD_PRO...,מיסוי ...,2009 ...,כן ...,תכנית מאושרת במימוש ...


,missing_percent
YachadMutza,0.852878
_id,0.000000
MisparMitham,0.000000
SemelYeshuv,0.000000
Yeshuv,0.000000
ShemMitcham,0.000000
YachadKayam,0.000000
YachadTosafti,0.000000
TaarichHachraza,0.000000
MisparTochnit,0.000000


Saved dataset inspection summary: data\metadata\dataset_inspection_summary.csv (1 rows)


,source_name,file_path,num_rows,num_columns,columns,missing_ratio_mean,has_city_like_column,has_street_like_column,has_plan_like_column,has_status_like_column,has_units_like_column,has_geometry_like_column
0,browser_export_f65a0daf-f737-49c5-9424-d378d52...,data\manual_sources\data_gov_f65a0daf_records....,938,17,"[""_id"", ""MisparMitham"", ""Yeshuv"", ""SemelYeshuv...",0.000502,False,False,False,True,False,True


## 9 - Create Initial Raw Combined Dataset and Preview

Create the main raw collected dataset from all successfully loaded relevant public sources. No standardization is forced here; schemas are combined with an outer join so Notebook 02 can clean and standardize them.

In [50]:
main_raw_collected_path = PROCESSED_DIR / "urban_renewal_raw_collected.csv"
preview_path = PROCESSED_DIR / "00_raw_combined_sources_preview.csv"
MAX_COMBINED_ROWS = 200_000
combined_df = pd.DataFrame()

if combined_frames:
    combined_df = pd.concat(combined_frames, ignore_index=True, join="outer")
    original_row_count = len(combined_df)
    was_truncated = False
    if len(combined_df) > MAX_COMBINED_ROWS:
        combined_df = combined_df.head(MAX_COMBINED_ROWS).copy()
        was_truncated = True

    combined_df.to_csv(main_raw_collected_path, index=False, encoding="utf-8-sig")
    combined_df.head(5000).to_csv(preview_path, index=False, encoding="utf-8-sig")

    print(f"Created raw collected urban renewal dataset at {main_raw_collected_path.relative_to(PROJECT_ROOT)}")
    print(f"Saved preview at {preview_path.relative_to(PROJECT_ROOT)}")
    print(f"Rows: {len(combined_df):,} of {original_row_count:,}; Columns: {combined_df.shape[1]:,}")
    if was_truncated:
        print(f"Combined dataset was truncated to {MAX_COMBINED_ROWS:,} rows for this MVP ingestion output.")
    print("Source counts:")
    print(combined_df["source_name"].value_counts(dropna=False).to_string())
    display(combined_df.head(10))
else:
    for stale_path in [main_raw_collected_path, preview_path]:
        if stale_path.exists():
            stale_path.unlink()
            print(f"Removed stale output: {stale_path.relative_to(PROJECT_ROOT)}")
    print("No valid strict urban-renewal datasets were collected. Review source_inventory.csv and candidate_resources.csv.")
    print("No fake rows were created.")


Created raw collected urban renewal dataset at data\processed\urban_renewal_raw_collected.csv
Saved preview at data\processed\00_raw_combined_sources_preview.csv
Rows: 938 of 938; Columns: 25
Source counts:
source_name
browser_export_f65a0daf-f737-49c5-9424-d378d52104f5    938


,_id,MisparMitham,Yeshuv,SemelYeshuv,ShemMitcham,YachadKayam,YachadTosafti,YachadMutza,TaarichHachraza,MisparTochnit,...,Bebitzua,Status,source_name,source_url,source_type,original_resource_id,original_file_path,original_row_index,ingestion_timestamp,detected_relevance_reason
0,1,4001,גבעתים ...,6300,ערבי נחל ...,126,108,530.0,20/08/2006,גב/490 ...,...,לא ...,תכנית מאושרת - אחרי רישוי ...,browser_export_f65a0daf-f737-49c5-9424-d378d52...,browser_saved_official_data_gov_json,browser_export_official_data_gov_api,f65a0daf-f737-49c5-9424-d378d52104f5,data\manual_sources\data_gov_f65a0daf_records....,0,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
1,2,4005,קרית אונו ...,2620,ישעיהו ...,198,198,396.0,20/08/2006,תממ/284 ...,...,כן ...,תכנית מאושרת במימוש ...,browser_export_f65a0daf-f737-49c5-9424-d378d52...,browser_saved_official_data_gov_json,browser_export_official_data_gov_api,f65a0daf-f737-49c5-9424-d378d52104f5,data\manual_sources\data_gov_f65a0daf_records....,1,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
2,3,4006,קרית אונו ...,2620,שאול המלך ...,180,48,531.0,27/05/2004,קא/מק/61/285/א ...,...,לא ...,תכנית מאושרת - אחרי רישוי ...,browser_export_f65a0daf-f737-49c5-9424-d378d52...,browser_saved_official_data_gov_json,browser_export_official_data_gov_api,f65a0daf-f737-49c5-9424-d378d52104f5,data\manual_sources\data_gov_f65a0daf_records....,2,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
3,4,4010,ראשון לציון ...,8300,רמת אליהו (פוזננסקי) ...,58,232,290.0,29/06/2017,413-0292680 ...,...,כן ...,תכנית מאושרת במימוש ...,browser_export_f65a0daf-f737-49c5-9424-d378d52...,browser_saved_official_data_gov_json,browser_export_official_data_gov_api,f65a0daf-f737-49c5-9424-d378d52104f5,data\manual_sources\data_gov_f65a0daf_records....,3,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
4,5,4011,ראשון לציון ...,8300,סלע ...,283,0,1386.0,09/02/2012,רצ/מק/1/13/19/4 ...,...,כן ...,תכנית מאושרת במימוש ...,browser_export_f65a0daf-f737-49c5-9424-d378d52...,browser_saved_official_data_gov_json,browser_export_official_data_gov_api,f65a0daf-f737-49c5-9424-d378d52104f5,data\manual_sources\data_gov_f65a0daf_records....,4,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
5,6,4012,רעננה ...,8700,ברנדיס ...,39,81,136.0,19/01/2017,רע/1/511 ...,...,לא ...,תכנית מאושרת - אחרי רישוי ...,browser_export_f65a0daf-f737-49c5-9424-d378d52...,browser_saved_official_data_gov_json,browser_export_official_data_gov_api,f65a0daf-f737-49c5-9424-d378d52104f5,data\manual_sources\data_gov_f65a0daf_records....,5,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
6,7,4017,תל אביב יפו ...,5000,רקאנטי ...,96,102,198.0,09/02/2012,תא/3850/מח ...,...,לא ...,תכנית מאושרת - אחרי רישוי ...,browser_export_f65a0daf-f737-49c5-9424-d378d52...,browser_saved_official_data_gov_json,browser_export_official_data_gov_api,f65a0daf-f737-49c5-9424-d378d52104f5,data\manual_sources\data_gov_f65a0daf_records....,6,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
7,8,4018,תל אביב יפו ...,5000,טאגור ...,60,84,144.0,09/02/2012,תא/3853/מח ...,...,לא ...,תכנית מאושרת - אחרי רישוי ...,browser_export_f65a0daf-f737-49c5-9424-d378d52...,browser_saved_official_data_gov_json,browser_export_official_data_gov_api,f65a0daf-f737-49c5-9424-d378d52104f5,data\manual_sources\data_gov_f65a0daf_records....,7,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
8,9,4024,תל אביב יפו ...,5000,הדר יוסף - לודג' ...,193,0,288.0,26/05/2011,תא/מק/3569 ...,...,לא ...,תכנית מאושרת לפני מימוש ...,browser_export_f65a0daf-f737-49c5-9424-d378d52...,browser_saved_official_data_gov_json,browser_export_official_data_gov_api,f65a0daf-f737-49c5-9424-d378d52104f5,data\manual_sources\data_gov_f65a0daf_records....,8,2026-05-26T15:05:10,Browser-saved official data.gov.il CKAN datast...
9,10,4025,קרית מלאכי ...,1034,בלוק 101 ...,40,45,195.0,06/04/2022,617-0379792 ...,...,לא ...,תכני

## 10 - Final Summary and Next Steps

Summarize what worked, what failed, which sources look promising, and what Notebook 02 should do next.

In [51]:
main_raw_collected_path = PROCESSED_DIR / "urban_renewal_raw_collected.csv"
final_check_df = pd.DataFrame()
final_rows = 0
final_csv_error = None

if main_raw_collected_path.exists():
    try:
        final_check_df = pd.read_csv(main_raw_collected_path)
        final_rows = len(final_check_df)
    except Exception as exc:
        final_csv_error = f"Could not read final CSV: {type(exc).__name__}: {exc}"
        final_rows = 0

worked = source_inventory_df[source_inventory_df["load_status"] == "loaded"] if not source_inventory_df.empty else pd.DataFrame()
failed = source_inventory_df[
    (source_inventory_df["download_status"] == "failed") | (source_inventory_df["load_status"] == "failed")
] if not source_inventory_df.empty else pd.DataFrame()
included = source_inventory_df[source_inventory_df["final_included_in_raw_csv"].fillna(False).astype(bool)] if not source_inventory_df.empty else pd.DataFrame()
promising = dataset_inspection_summary_df[
    dataset_inspection_summary_df[
        [
            "has_city_like_column",
            "has_street_like_column",
            "has_plan_like_column",
            "has_status_like_column",
            "has_units_like_column",
            "has_geometry_like_column",
        ]
    ].any(axis=1)
] if not dataset_inspection_summary_df.empty else pd.DataFrame()

api_diagnostics_path = METADATA_DIR / "api_diagnostics.csv"
api_diag = pd.read_csv(api_diagnostics_path) if api_diagnostics_path.exists() else pd.DataFrame(columns=API_DIAGNOSTICS_COLUMNS)
package_search_attempts = api_diag[api_diag["endpoint"] == DATA_GOV_CKAN_ENDPOINT] if not api_diag.empty and "endpoint" in api_diag else pd.DataFrame()
package_search_success_count = int(package_search_attempts["parsed_json_success"].fillna(False).astype(bool).sum()) if not package_search_attempts.empty else 0
package_search_failure_count = int(len(package_search_attempts) - package_search_success_count) if not package_search_attempts.empty else 0
package_search_status = "success" if package_search_success_count else "failed_or_not_available"

known_attempted = len(MANUAL_KNOWN_RESOURCE_IDS)
known_loaded_successfully = sum(1 for item in known_resource_results if item.get("success"))
known_rows_by_resource = pd.DataFrame(known_resource_results)

strict_candidate_count = int((candidates_df["candidate_decision"] == "include_for_download").sum()) if not candidates_df.empty and "candidate_decision" in candidates_df else 0
broad_only_candidate_count = int((candidates_df["candidate_decision"] == "metadata_only").sum()) if not candidates_df.empty and "candidate_decision" in candidates_df else 0
resources_downloaded = int((source_inventory_df["download_status"] == "downloaded").sum()) if not source_inventory_df.empty else 0
loaded_datastore = int(source_inventory_df["load_method"].isin(["ckan_datastore_search", "ckan_datastore_search_paginated"]).sum()) if not source_inventory_df.empty else 0
loaded_direct = int((source_inventory_df["load_method"] == "direct_download").sum()) if not source_inventory_df.empty else 0
invalid_html_rejected = int(source_inventory_df["detected_content_kind"].isin(["html", "blocked_or_security_page"]).sum()) if not source_inventory_df.empty else 0
valid_included_count = int(source_inventory_df["final_included_in_raw_csv"].fillna(False).astype(bool).sum()) if not source_inventory_df.empty else 0

print("Final summary")
print("=" * 80)
print(f"Package search status: {package_search_status}")
print(f"Package search API attempts: {len(package_search_attempts):,}; successes: {package_search_success_count:,}; failures: {package_search_failure_count:,}")
print(f"Total CKAN search results: {len(search_results_df):,}")
print(f"Strict urban renewal candidates from package_search: {strict_candidate_count:,}")
print(f"Broad-only candidates kept as metadata only: {broad_only_candidate_count:,}")
print(f"Known resource IDs attempted: {known_attempted:,}")
print(f"Known resource IDs loaded successfully: {known_loaded_successfully:,}")
if not known_rows_by_resource.empty:
    print("Rows loaded per known resource:")
    print(known_rows_by_resource[["source_name", "resource_id", "success", "rows_loaded", "total_records_reported", "error_message"]].to_string(index=False))
print(f"Resources downloaded/API responses saved: {resources_downloaded:,}")
print(f"Resources loaded through datastore_search: {loaded_datastore:,}")
print(f"Resources loaded through direct download: {loaded_direct:,}")
print(f"Invalid HTML/security pages rejected: {invalid_html_rejected:,}")
print(f"Valid datasets included in raw CSV: {valid_included_count:,}")
print(f"Rows in final urban_renewal_raw_collected.csv: {final_rows:,}")

if final_csv_error:
    print(final_csv_error)

if not final_check_df.empty and "source_name" in final_check_df.columns:
    print("\nSource counts in final CSV:")
    print(final_check_df["source_name"].value_counts(dropna=False).to_string())
else:
    print("\nNo valid strict urban-renewal datasets were collected. Review source_inventory.csv and candidate_resources.csv.")

print("\n1. Sources worked and passed loading:")
if not worked.empty:
    display_columns = [col for col in ["source_name", "original_resource_id", "file_path", "load_method", "num_rows", "num_columns", "final_included_in_raw_csv"] if col in worked.columns]
    print(worked[display_columns].to_string(index=False))
else:
    print("No datasets loaded successfully.")

print("\n2. Sources failed, were rejected, or need follow-up:")
if not failed.empty:
    display_columns = [col for col in ["source_name", "original_resource_id", "source_url", "download_status", "load_status", "detected_content_kind", "error_message"] if col in failed.columns]
    print(failed[display_columns].head(25).to_string(index=False))
else:
    print("No failed downloads or loads were recorded.")

print("\n3. Sources that look promising for Notebook 02:")
if not included.empty:
    display_columns = [col for col in ["source_name", "original_resource_id", "num_rows", "num_columns", "load_method", "matched_strict_keywords"] if col in included.columns]
    print(included[display_columns].to_string(index=False))
else:
    print("No source passed the strict urban-renewal final inclusion gates.")

print("\n4. Relevant columns observed:")
if not dataset_inspection_summary_df.empty:
    all_columns = []
    for cols_json in dataset_inspection_summary_df["columns"].dropna():
        try:
            all_columns.extend(json.loads(cols_json))
        except Exception:
            pass
    relevant_columns = sorted({col for col in all_columns if text_contains_any(col, CITY_KEYWORDS + STREET_KEYWORDS + PLAN_KEYWORDS + STATUS_KEYWORDS + UNITS_KEYWORDS + GEOMETRY_KEYWORDS)})
    print(relevant_columns if relevant_columns else "No obvious relevant columns detected by keyword rules.")
else:
    print("No loaded datasets available for column review.")

print("\n5. Recommended next steps for Notebook 02:")
print("- Read data/processed/urban_renewal_raw_collected.csv only if it exists and has rows.")
print("- Profile source-specific columns before mapping to the target schema.")
print("- Standardize city, street/area, plan number, renewal type, planning status, unit counts, and source metadata.")
print("- Preserve raw values and add confidence_level and lawyer_note fields without inventing missing data.")

if main_raw_collected_path.exists() and final_rows > 0:
    print(f"\nCreated raw collected urban renewal dataset at {main_raw_collected_path.relative_to(PROJECT_ROOT)}")
else:
    print("No valid urban-renewal dataset CSV exists yet. Notebook 02 should not be started.")


Final summary
Package search status: failed_or_not_available
Package search API attempts: 22; successes: 0; failures: 22
Total CKAN search results: 0
Strict urban renewal candidates from package_search: 0
Broad-only candidates kept as metadata only: 0
Known resource IDs attempted: 3
Known resource IDs loaded successfully: 0
Rows loaded per known resource:
                                    source_name                          resource_id  success  rows_loaded total_records_reported                                                       error_message
            data.gov.il - מתחמי התחדשות עירונית f65a0daf-f737-49c5-9424-d378d52104f5    False            0                   None Downloaded content appears to be HTML/security page, not a dataset.
data.gov.il - מתחמי התחדשות עירונית מוכרזים GIS ceb7bbb0-e2db-4e87-8a6c-0a250f5de001    False            0                   None Downloaded content appears to be HTML/security page, not a dataset.
  data.gov.il - תוכניות מתאר של התחדשות עירונית 